# Khảo sát dữ liệu RetailRocket (EDA) — Đề tài 3 `bd03_khuyen_nghi`

Notebook khảo sát dữ liệu gốc trước khi thiết kế pipeline (mục A.5 và mốc tuần 2 của đề cương).

**Cách tạo số liệu.** Mọi phép đếm chạy bằng Spark trên cụm, trên 4 tệp CSV gốc của Kaggle ở
`hdfs:///lakehouse/bd03_khuyen_nghi/landing/retailrocket/` (chưa qua bronze, chưa thêm 5 % người dùng tổng hợp):

```bash
bash ha_tang/spark/submit.sh lo kiem_tra/eda.py | tail -1 > docs/bang_chung/cum/eda.json
```

`scripts/thu_bang_chung.sh` chạy sẵn lệnh này (bước 8). Job chỉ đưa về driver các bảng tổng hợp nhỏ
(theo ngày, theo giờ, theo nhóm hoạt động), nên notebook này chạy được trên máy bất kỳ chỉ với pandas + matplotlib.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

TEP = Path("../docs/bang_chung/cum/eda.json")
if not TEP.exists():
    raise SystemExit("Chưa có {}: chạy kiem_tra/eda.py trên cụm (xem ô đầu).".format(TEP))
dong = [d for d in TEP.read_text(encoding="utf-8").splitlines() if d.strip().startswith("{")]
eda = json.loads(dong[-1])
plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.3})
tq = eda["tong_quan"]
pd.Series(tq, name="giá trị").to_frame()

## 1. Loại sự kiện và phễu chuyển đổi

In [ ]:
loai = pd.DataFrame(eda["theo_loai"]).set_index("event")
loai["ty_le_su_kien"] = loai["so_su_kien"] / loai["so_su_kien"].sum()
display(loai)
ph = eda["pheu_nguoi_dung"]
print("Người dùng có xem: {:,} | có thêm giỏ: {:,} ({:.2%}) | có mua: {:,} ({:.2%})".format(
    ph["co_xem"], ph["co_them_gio"], ph["co_them_gio"] / ph["co_xem"], ph["co_mua"], ph["co_mua"] / ph["co_xem"]))
ax = loai["so_su_kien"].plot.bar(logy=True, title="Số sự kiện theo loại (thang log)", rot=0, color="#2563eb")
ax.set_ylabel("sự kiện"); plt.show()

## 2. Theo thời gian

In [ ]:
ngay = pd.DataFrame(eda["theo_ngay"])
ngay["ngay"] = pd.to_datetime(ngay["ngay"])
ax = ngay.plot(x="ngay", y=["so_su_kien", "so_nguoi_dung"], title="Sự kiện và người dùng hoạt động mỗi ngày")
ax2 = ax.twinx(); ax2.plot(ngay["ngay"], ngay["so_mua"], color="#15803d", alpha=0.6, label="số giao dịch"); ax2.set_ylabel("giao dịch")
ax2.grid(False); ax2.legend(loc="upper right"); plt.show()
gio = pd.DataFrame(eda["theo_gio_utc"]).set_index("gio")["count"]
gio.plot.bar(title="Sự kiện theo giờ trong ngày (UTC)", rot=0, color="#d97706"); plt.show()
print("Từ {} đến {} ({} ngày).".format(tq["tu_ngay"], tq["den_ngay"], tq["so_ngay"]))

## 3. Đuôi dài: người dùng và sản phẩm

In [ ]:
fig, truc = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, ten, mau in ((truc[0], "nguoi_dung", "#7c3aed"), (truc[1], "san_pham", "#0891b2")):
    pb = pd.DataFrame(eda["phan_bo_" + ten]).set_index("nhom")["so_luong"]
    pb.plot.bar(ax=ax, logy=True, rot=45, color=mau, title="Số {} theo số sự kiện (log)".format(ten.replace("_", " ")))
plt.tight_layout(); plt.show()
it = eda["nguoi_dung_it_hoat_dong"]
display(pd.DataFrame({"người dùng": eda["phan_vi_nguoi_dung"], "sản phẩm": eda["phan_vi_san_pham"]}))
print("Người dùng chỉ có 1 sự kiện: {:,} ({:.1%}); dưới 3 sự kiện: {:.1%}; từ 3 sự kiện trở lên: {:,}".format(
    it["chi_1_su_kien"], it["chi_1_su_kien"] / tq["so_nguoi_dung"], it["duoi_3_su_kien"] / tq["so_nguoi_dung"],
    it["tu_3_su_kien"]))
tt = eda["tap_trung_san_pham"]
print("1 % sản phẩm phổ biến nhất chiếm {:.1%} sự kiện; 10 % phổ biến nhất chiếm {:.1%}.".format(tt["top_1pct"], tt["top_10pct"]))

## 4. Chất lượng dữ liệu

In [ ]:
cl = pd.Series(eda["chat_luong"], name="số lượng").to_frame()
cl["tỷ lệ trên tổng sự kiện"] = [v / tq["so_su_kien"] if k.startswith(("dong", "nhap", "so_su_kien", "mua", "khong", "thieu")) else None
                                 for k, v in eda["chat_luong"].items()]
cl

## 5. Thuộc tính sản phẩm và cây danh mục

In [ ]:
display(pd.Series(eda.get("thuoc_tinh", {}), name="giá trị").to_frame())
display(pd.Series(eda.get("cay_danh_muc", {}), name="giá trị").to_frame())

## 6. Hệ quả cho thiết kế pipeline

Các nhận xét dưới đây được tính từ chính số liệu ở trên (chạy lại notebook với dữ liệu khác thì câu chữ đổi theo).

In [ ]:
cl, th = eda["chat_luong"], eda.get("thuoc_tinh", {})
nhan_xet = [
    "Ma trận người dùng × sản phẩm có mật độ {:.1e} ({:,} cặp trên {:,} × {:,}): phản hồi ngầm, cực thưa "
    "→ ALS implicitPrefs=true với trọng số xem/thêm giỏ/mua.".format(tq["mat_do_ma_tran"], tq["so_cap_nguoi_san_pham"],
                                                                     tq["so_nguoi_dung"], tq["so_san_pham"]),
    "{:.1%} người dùng chỉ có 1 sự kiện, {:.1%} dưới 3 sự kiện → lọc ≥ 3 tương tác khi huấn luyện (mục 3.10) và bắt buộc "
    "có chiến lược cold start cho phần còn lại.".format(it["chi_1_su_kien"] / tq["so_nguoi_dung"],
                                                       it["duoi_3_su_kien"] / tq["so_nguoi_dung"]),
    "1 % sản phẩm chiếm {:.1%} sự kiện → popularity bias rõ: phải đo coverage, novelty và thử xếp lại theo novelty."
    .format(tt["top_1pct"]),
    "{:,} dòng trùng hoàn toàn và {:,} lần nhấp lặp trong 5 giây → bronze_to_silver khử trùng theo mã sự kiện, theo nội dung "
    "và theo cửa sổ 5 giây.".format(cl["dong_trung_hoan_toan"], cl["nhap_lap_trong_5_giay"]),
    "{:,} người dùng có giờ vượt 500 sự kiện ({:,} sự kiện, giờ đông nhất {:,}) → lọc bot theo ngưỡng của đặc tả."
    .format(cl["so_nguoi_dung_bot"], cl["so_su_kien_cua_bot"], cl["gio_nhieu_su_kien_nhat_cua_1_nguoi"]),
    "Dữ liệu trải {} ngày ({} → {}) → chia train/test theo thời gian, tháng cuối làm test; silver phân vùng theo ngày "
    "(khoảng {} phân vùng, không phân vùng theo người dùng vì có {:,} người dùng).".format(
        tq["so_ngay"], tq["tu_ngay"], tq["den_ngay"], tq["so_ngay"], tq["so_nguoi_dung"]),
]
if th:
    nhan_xet.append("{:,}/{:,} sản phẩm có danh mục, {:,} có thuộc tính 790 (giá ẩn danh) → khối danh mục cho cold start và "
                    "ràng buộc tầm giá của mô hình v2 dùng được, nhưng giá chỉ có ý nghĩa so sánh.".format(
                        th["san_pham_co_danh_muc"], th["so_san_pham"], th["san_pham_co_gia_790"]))
for i, d in enumerate(nhan_xet, 1):
    print("{}. {}".format(i, d))